In [0]:
import mlflow.sklearn
import pandas as pd
from pyspark.sql.functions import lit

# Load model from registry
model_uri = "models:/workspace.default.fraud_classifier/1"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Load Gold table for scoring
df_score = spark.table("fraud_gold").toPandas()
X_score = df_score.drop(columns=["Class"])

# Score
df_score["fraud_probability"] = loaded_model.predict_proba(X_score)[:, 1]
df_score["fraud_prediction"] = loaded_model.predict(X_score)

# Write scored results back to Delta
spark.createDataFrame(df_score) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fraud_scored")

print("Batch scoring complete.")
print(f"Total scored: {spark.table('fraud_scored').count()}")

# Spot check high risk
spark.table("fraud_scored") \
    .filter("fraud_probability > 0.8") \
    .select("fraud_probability", "fraud_prediction", "Class", "Amount") \
    .orderBy("fraud_probability", ascending=False) \
    .show(20)

Batch scoring complete.
Total scored: 283726
+------------------+----------------+-----+------+
| fraud_probability|fraud_prediction|Class|Amount|
+------------------+----------------+-----+------+
|               1.0|               1|    1| 99.99|
|               1.0|               1|    1| 11.39|
|               1.0|               1|    1|  88.0|
|               1.0|               1|    1|179.66|
|0.9997669734981106|               1|    1|   1.0|
|0.9997171639765984|               1|    1| 648.0|
|0.9997049616016569|               1|    1|130.44|
|0.9996910341111436|               1|    1|364.19|
|0.9996900630554295|               1|    1|   1.0|
| 0.999681839105182|               1|    1| 227.3|
|0.9996781840146496|               1|    1|   1.0|
|0.9996686467706726|               1|    1|   1.0|
| 0.999664244761826|               1|    1|   1.0|
|0.9996611749443668|               1|    1|113.92|
|0.9996527106673276|               1|    1|   1.0|
|0.9996518332387324|               1|